# ELEN4025 – Machine Learning Group Project
## Stage 1: Data Loading & Sanity Checks

In [17]:
pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
#Set Up
import pandas as pd
import numpy as np
import os
import subprocess
import zipfile

RAW_DIR = os.path.join("data", "raw")
PROCESSED_DIR = os.path.join("data", "processed")
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

EXPECTED_TABLES = {
    "studentInfo":         {"min_rows": 30000, "key_cols": ["code_module", "code_presentation", "id_student"]},
    "studentVle":          {"min_rows": 10000000, "key_cols": ["code_module", "code_presentation", "id_student", "id_site", "date"]},
    "assessments":         {"min_rows": 200, "key_cols": ["code_module", "code_presentation", "id_assessment"]},
    "studentAssessment":   {"min_rows": 170000, "key_cols": ["id_assessment", "id_student"]},
    "studentRegistration": {"min_rows": 30000, "key_cols": ["code_module", "code_presentation", "id_student"]},
    "courses":             {"min_rows": 20, "key_cols": ["code_module", "code_presentation"]},
    "vle":                 {"min_rows": 6000, "key_cols": ["id_site", "code_module", "code_presentation"]},
}

print("Configuration ready.")
print(f"  RAW_DIR:       {RAW_DIR}")
print(f"  PROCESSED_DIR: {PROCESSED_DIR}")
print(f"  Expected tables: {list(EXPECTED_TABLES.keys())}")

Configuration ready.
  RAW_DIR:       data\raw
  PROCESSED_DIR: data\processed
  Expected tables: ['studentInfo', 'studentVle', 'assessments', 'studentAssessment', 'studentRegistration', 'courses', 'vle']


In [19]:
#Download and Extract Dataset 
ZIP_PATH = os.path.join("data", "oulad.zip")

if os.path.exists(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(RAW_DIR)
        extracted = zf.namelist()
    print(f"Extracted {len(extracted)} files from {ZIP_PATH}:")
    for f in sorted(extracted):
        print(f"  {f}")
else:
    print(f"Zip file not found at {ZIP_PATH}.")
    print("Checking if CSVs already exist in data/raw/ ...")
    existing = [f for f in os.listdir(RAW_DIR) if f.endswith(".csv")]
    print(f"  Found {len(existing)} CSV files: {existing}")

Extracted 7 files from data\oulad.zip:
  assessments.csv
  courses.csv
  studentAssessment.csv
  studentInfo.csv
  studentRegistration.csv
  studentVle.csv
  vle.csv


In [20]:
#Check all CSV files have been exraccted
expected_files = [f"{name}.csv" for name in EXPECTED_TABLES]
for fname in expected_files:
    path = os.path.join(RAW_DIR, fname)
    assert os.path.exists(path), f"FAIL: Missing file {path}"
    size_mb = os.path.getsize(path) / 1e6
    print(f"   {fname:30s} ({size_mb:.1f} MB)")

print(f"\n All {len(expected_files)} CSV files present.")

   studentInfo.csv                (3.5 MB)
   studentVle.csv                 (453.8 MB)
   assessments.csv                (0.0 MB)
   studentAssessment.csv          (5.7 MB)
   studentRegistration.csv        (1.1 MB)
   courses.csv                    (0.0 MB)
   vle.csv                        (0.3 MB)

 All 7 CSV files present.


In [21]:
# Load CSV Tables 
tables = {}
for name in EXPECTED_TABLES:
    path = os.path.join(RAW_DIR, f"{name}.csv")
    tables[name] = pd.read_csv(path)
    print(f"Loaded {name:25s} -> {tables[name].shape[0]:>10,} rows x {tables[name].shape[1]:>3} cols")

print(f"\nAll {len(tables)} tables loaded.")

Loaded studentInfo               ->     32,593 rows x  12 cols
Loaded studentVle                -> 10,655,280 rows x   6 cols
Loaded assessments               ->        206 rows x   6 cols
Loaded studentAssessment         ->    173,912 rows x   5 cols
Loaded studentRegistration       ->     32,593 rows x   5 cols
Loaded courses                   ->         22 rows x   3 cols
Loaded vle                       ->      6,364 rows x   6 cols

All 7 tables loaded.


In [22]:
#Check and confirm all tables have been loaded correctly 
assert len(tables) == 7, f"FAIL: Expected 7 tables, got {len(tables)}"
print("All 7 tables loaded into memory.")

for name, df in tables.items():
    assert len(df) > 0, f"FAIL: {name} is empty"
print("No empty tables.")

for name, df in tables.items():
    assert df.shape[0] >= EXPECTED_TABLES[name]["min_rows"], (
        f"FAIL: {name} has {df.shape[0]} rows, expected >= {EXPECTED_TABLES[name]['min_rows']}"
    )
print("All row counts in expected range.")



All 7 tables loaded into memory.
No empty tables.
All row counts in expected range.


In [23]:
#Inspect Data Types and Shapes
for name, df in tables.items():
    print(f"\n{'─'*55}")
    print(f"  {name}  |  {df.shape[0]:,} rows x {df.shape[1]} cols")
    print(f"{'─'*55}")
    print(df.dtypes.to_string())
    print(f"\nFirst 3 rows:")
    print(df.head(3).to_string())


───────────────────────────────────────────────────────
  studentInfo  |  32,593 rows x 12 cols
───────────────────────────────────────────────────────
code_module               str
code_presentation         str
id_student              int64
gender                    str
region                    str
highest_education         str
imd_band                  str
age_band                  str
num_of_prev_attempts    int64
studied_credits         int64
disability                str
final_result              str

First 3 rows:
  code_module code_presentation  id_student gender                region      highest_education imd_band age_band  num_of_prev_attempts  studied_credits disability final_result
0         AAA             2013J       11391      M   East Anglian Region       HE Qualification  90-100%     55<=                     0              240          N         Pass
1         AAA             2013J       28400      F              Scotland       HE Qualification   20-30%    35-55     

In [24]:
#Verify Key Columns Exist with correct types 
for name, df in tables.items():
    for col in EXPECTED_TABLES[name]["key_cols"]:
        assert col in df.columns, f"FAIL: {name} missing column '{col}'"
print("All key columns present in every table.")

# Check specific expected column counts
assert tables["studentInfo"].shape[1] == 12, "FAIL: studentInfo should have 12 columns"
assert tables["studentVle"].shape[1] == 6, "FAIL: studentVle should have 6 columns"
assert tables["courses"].shape[1] == 3, "FAIL: courses should have 3 columns"
assert tables["assessments"].shape[1] == 6, "FAIL: assessments should have 6 columns"
assert tables["studentAssessment"].shape[1] == 5, "FAIL: studentAssessment should have 5 columns"
assert tables["studentRegistration"].shape[1] == 5, "FAIL: studentRegistration should have 5 columns"
assert tables["vle"].shape[1] == 6, "FAIL: vle should have 6 columns"
print(" All Column counts correct for all tables.")

# sum_click must be numeric for aggregation; id_student must be numeric for joins.
# If pandas read them as strings, downstream computations would silently fail.
assert tables["studentVle"]["sum_click"].dtype in [np.int64, np.int32], "FAIL: sum_click should be integer"
assert tables["studentInfo"]["id_student"].dtype in [np.int64, np.int32], "FAIL: id_student should be integer"
print("Critical columns have expected dtypes.")



All key columns present in every table.
 All Column counts correct for all tables.
Critical columns have expected dtypes.


## Missing/ Null Vlue Analysis 

In [25]:
#Identify Missing Data 
print("Missing value analysis per table:\n")

missing_summary = []
for name, df in tables.items():
    total_missing = df.isnull().sum().sum()
    if total_missing == 0:
        print(f"  {name}: ✓ no missing values")
    else:
        miss = df.isnull().sum()
        miss = miss[miss > 0]
        for col, count in miss.items():
            pct = 100.0 * count / len(df)
            missing_summary.append({
                "Table": name,
                "Column": col,
                "Missing Count": count,
                "Missing %": round(pct, 2),
            })
            print(f"  {name}.{col}: {count:,} missing ({pct:.2f}%)")

Missing value analysis per table:

  studentInfo.imd_band: 1,111 missing (3.41%)
  studentVle: ✓ no missing values
  assessments.date: 11 missing (5.34%)
  studentAssessment.score: 173 missing (0.10%)
  studentRegistration.date_registration: 45 missing (0.14%)
  studentRegistration.date_unregistration: 22,521 missing (69.10%)
  courses: ✓ no missing values
  vle.week_from: 5,243 missing (82.39%)
  vle.week_to: 5,243 missing (82.39%)


In [26]:
#Table shwoing missing values summary
missing_df = pd.DataFrame(missing_summary)
print("\n─── Missing Values Summary Table ───\n")
print(missing_df.to_string(index=False))


─── Missing Values Summary Table ───

              Table              Column  Missing Count  Missing %
        studentInfo            imd_band           1111       3.41
        assessments                date             11       5.34
  studentAssessment               score            173       0.10
studentRegistration   date_registration             45       0.14
studentRegistration date_unregistration          22521      69.10
                vle           week_from           5243      82.39
                vle             week_to           5243      82.39


In [30]:
#Document Missing Values and Confirm Critical Columns are Clean
si = tables["studentInfo"]

for col in ["id_student", "final_result", "code_module", "code_presentation", "gender", "region"]:
    assert si[col].isnull().sum() == 0, f"FAIL: studentInfo.{col} has nulls"
print("No missing values in critical studentInfo columns.")


assert tables["studentVle"].isnull().sum().sum() == 0, "FAIL: studentVle has nulls"
print("studentVle has zero missing values.")


assert tables["courses"].isnull().sum().sum() == 0, "FAIL: courses has nulls"
print("courses has zero missing values.")


assert si["imd_band"].isnull().sum() == 1111, "FAIL: imd_band missing count unexpected"
print("imd_band: 1,111 missing.")

unreg_missing = tables["studentRegistration"]["date_unregistration"].isnull().sum()
assert unreg_missing == 22521, f"FAIL: date_unregistration missing count unexpected: {unreg_missing}"
print(f"date_unregistration: {unreg_missing:,} missing.")

score_missing = tables["studentAssessment"]["score"].isnull().sum()
assert score_missing == 173, f"FAIL: score missing count unexpected: {score_missing}"
print(f"score: {score_missing} missing.")

print("\n All missing values documented.")

No missing values in critical studentInfo columns.
studentVle has zero missing values.
courses has zero missing values.
imd_band: 1,111 missing.
date_unregistration: 22,521 missing.
score: 173 missing.

 All missing values documented.


In [ ]:
#Identify and remove duplicate rows 
print("Duplicate analysis per table:\n")
print(f"{'Table':25s} {'Duplicates':>12s}   {'Decision':30s}")
print("─" * 75)

for name, df in tables.items():
    n_dup = df.duplicated().sum()
    if name == "studentVle" and n_dup > 0:
        decision = "KEEP"
    elif n_dup > 0:
        decision = "REMOVE"
    else:
        decision = "OK — none found"
    print(f"{name:25s} {n_dup:>12,}   {decision}")


Duplicate analysis per table:

Table                       Duplicates   Decision                      
───────────────────────────────────────────────────────────────────────────
studentInfo                          0   OK — none found
studentVle                     787,170   KEEP — aggregate in Stage 3
assessments                          0   OK — none found
studentAssessment                    0   OK — none found
studentRegistration                  0   OK — none found
courses                              0   OK — none found
vle                                  0   OK — none found
